# PCA·다변량 요약 (모든 숫자 변수 동시)

## 개념 정리

- **상관계수 / 산점도**는 보통 **두 변수**쌍에 대한 선형 관계를 봅니다. (변수가 많으면 행렬로 전부 볼 수는 있음)
- **PCA(주성분 분석)**는 “두 변수만”이 아니라 **변수가 $p$개면 $p$개를 한꺼번에** 넣어, 분산이 큰 방향(제1주성분), 그와 수직인 다음 방향(제2주성분)… 을 순서대로 찾습니다. 그래서 **3개, 10개, 72개**든 상관없이 **한 번에** 차원을 줄인 축을 얻습니다.
- 변수가 **3개**이면 이론상 주성분은 **최대 3개**(공간 전체). **시각화**는 보통 PC1–PC2 **2차원** 또는 PC1–PC2–PC3 **3차원** 산점도로 합니다.
- 그 밖에 **3개 이상**에서 쓸 수 있는 것: **상관 행렬 히트맵**, **다중공선성(VIF)**, **인자분석**, **t-SNE/UMAP**(비선형), **페어플롯**(여러 쌍 동시) 등이 있습니다.

아래는 `train.csv`의 **숫자형 컬럼**에 대해 표준화 후 PCA를 적합합니다. (샘플로 속도 조절 가능)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 — 3D 축 등록

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

SEED = 42
np.random.seed(SEED)

# 전체 데이터가 무거우면 행만 샘플링 (None이면 전체)
SAMPLE_N = 80_000

from sklearn.model_selection import train_test_split

train = pd.read_csv('train.csv')
if SAMPLE_N is not None and len(train) > SAMPLE_N:
    train, _ = train_test_split(
        train, train_size=SAMPLE_N, stratify=train['fraud_bool'], random_state=SEED
    )

TARGET = 'fraud_bool'
drop_cols = {'id', TARGET}
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c not in drop_cols]

X = train[num_cols].values
y = train[TARGET].values

X = SimpleImputer(strategy='median').fit_transform(X)
X = StandardScaler().fit_transform(X)

n_comp = min(len(num_cols), X.shape[0])  # 이론적 상한
pca = PCA(n_components=min(20, n_comp), random_state=SEED)
Z = pca.fit_transform(X)

print('숫자 피처 수:', len(num_cols))
print('PCA 성분 수:', Z.shape[1])
print('분산 설명비율(앞 5개):', np.round(pca.explained_variance_ratio_[:5], 4))
print('누적 분산(앞 5개):', np.round(np.cumsum(pca.explained_variance_ratio_[:5]), 4))

: 

## “모든 변수의 한 선형관계”를 한 장에?

- **단일 숫자 하나**로 “전체 변수가 가진 선형관계”를 표현하는 것은 없습니다. 변수가 $p$개면 선형 구조는 보통 **공분산/상관 행렬 ($p \times p$)** 또는 **여러 주성분**으로 표현됩니다.
- **가장 가까운 ‘한 줄’ 요약**: **제1주성분 PC1** = 변수들에 대한 **하나의 가중합** $w_1 x_1 + \cdots + w_p x_p$. 가중치(loading)가 “여러 변수가 함께 움직이는” 방향을 한 번에 보여 줍니다.
- **모든 쌍의 선형 관계**를 한 번에: **상관 행렬 히트맵** (아래 별도 셀).
- **각 주성분에 변수가 얼마나 기여하는지** 한 번에: **로딩 히트맵** (다음 셀).

In [ ]:
# 주성분 로딩: PC1~PCk = 각 변수의 선형 가중치 (행: PC, 열: 원 변수)
k_load = min(5, pca.components_.shape[0])
load = pd.DataFrame(
    pca.components_[:k_load],
    index=[f'PC{i+1}' for i in range(k_load)],
    columns=num_cols,
)
fig, ax = plt.subplots(figsize=(max(10, len(num_cols) * 0.25), 4))
im = ax.imshow(load.values, aspect='auto', cmap='RdBu_r', vmin=-0.7, vmax=0.7)
ax.set_xticks(range(len(num_cols)))
ax.set_xticklabels(num_cols, rotation=90, fontsize=7)
ax.set_yticks(range(k_load))
ax.set_yticklabels(load.index)
ax.set_title('PCA loadings (표준화된 변수 → 각 PC에 대한 기여)')
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

: 

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ev = pca.explained_variance_ratio_
ax.bar(range(1, len(ev) + 1), ev, alpha=0.7, label='각 PC')
ax.plot(range(1, len(ev) + 1), np.cumsum(ev), 'ro-', markersize=4, label='누적')
ax.set_xlabel('주성분 번호')
ax.set_ylabel('분산 비율')
ax.legend()
ax.set_title('PCA 분산 설명비율')
plt.tight_layout()
plt.show()

: 

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
mask = y == 1
ax.scatter(Z[~mask, 0], Z[~mask, 1], s=2, alpha=0.25, c='steelblue', label='정상')
ax.scatter(Z[mask, 0], Z[mask, 1], s=6, alpha=0.6, c='crimson', label='사기')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('PC1 vs PC2 (전체 숫자 피처 PCA)')
ax.legend(markerscale=3)
plt.tight_layout()
plt.show()

### 3개 주성분 (PC1, PC2, PC3)

원래 피처가 3개든 30개든, **투영된 좌표** 3개(PC1~PC3)로 3D 산점도를 그릴 수 있습니다.

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
mask = y == 1
ax.scatter(Z[~mask, 0], Z[~mask, 1], Z[~mask, 2], s=2, alpha=0.2, c='steelblue', label='정상')
ax.scatter(Z[mask, 0], Z[mask, 1], Z[mask, 2], s=8, alpha=0.5, c='crimson', label='사기')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')
ax.set_title('PC1–PC2–PC3')
ax.legend()
plt.tight_layout()
plt.show()

### (참고) 모든 변수 쌍의 선형 상관 — 히트맵

PCA와 별개로, **두 변수씩** 선형 상관을 전부 보려면 상관 행렬을 그립니다.

In [ ]:
corr = train[num_cols].corr()
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr.values, aspect='auto', cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(num_cols)))
ax.set_yticks(range(len(num_cols)))
ax.set_xticklabels(num_cols, rotation=90, fontsize=6)
ax.set_yticklabels(num_cols, fontsize=6)
ax.set_title('숫자 피처 상관 행렬')
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()